In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
pip install cohere qdrant-client

   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   ------------------- -------------------- 2.4/4.9 MB 15.8 MB/s eta 0:00:01
   --------------------------- ------------ 3.4/4.9 MB 7.7 MB/s eta 0:00:01
   ---------------------------------- ----- 4.2/4.9 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 4.9/4.9 MB 5.9 MB/s  0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.9 MB ? eta -:--:--
   ------ --------------------------------- 1.0/6.9 MB 2.7 MB/s eta 0:00:03
   --------- ------------------------------ 1.6/6.9 MB 2.9 MB/s eta 0:00:02
   --------------- ------------------------ 2.6/6.9 MB 3.3 MB/s eta 0:00:02
   ------------------- -------------------- 3.4/6.9 MB 3.5 MB/s eta 0:00:02
   ------------------------- -------------- 4.5/6.9 MB 3.8 MB/s eta 0:00:01
   ------------------------------- -------- 5.5/6.9 MB 4.1 MB/s eta 0:00:01
   --------------------------------

In [7]:
import os
import base64
import pymupdf
from PIL import Image
from langchain_core.embeddings import Embeddings
import cohere

In [4]:
def pdf_data_to_images(path,max_pages = None):
    docs= pymupdf.open(path)
    length = len(docs)

    if max_pages:
        length = min(length,max_pages)

    urls = []
    pngs = []

    for i in range(length):
        page = docs[i]
        png = page.get_pixmap().tobytes("png")
        b64 = base64.b64encode(png).decode()
        image_url = f"data:image/png;base64,{b64}"

        pngs.append(png)
        urls.append(image_url)

    docs.close()
    return urls,pngs

page_urls, page_pngs = pdf_data_to_images("../data/pdf/medical_report.pdf")

    

In [10]:
class CohereEmbeddings(Embeddings):
    def __init__(self, model = "embed-v4.0"):
        self.client = cohere.ClientV2(api_key= os.getenv("COHERE_API_KEY"))
        self.model = model

    def embed_documents(self , images):
        result = self.client.embed(
            model= self.model,
            input_type = "search_document",
            images=images,
            embedding_types=["float"]
        )

        return [list(value) for value in result.embeddings.float_]

    def embed_query(self, text):
        result = self.client.embed(
            model= self.model,
            input_type = "search_query",
            texts=[text],
            embedding_types=["float"]
        )

        return list(result.embeddings.float_)

### Generate Embeddings

In [11]:
embeddings = CohereEmbeddings()
docEmbeddings = embeddings.embed_documents(page_urls)

In [12]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

In [13]:
DIM = len(docEmbeddings[0])

In [14]:
client = QdrantClient(
    location=":memory:"
)

In [15]:
client.create_collection(
    collection_name="pdf_pages",
    vectors_config=VectorParams(
        size=DIM,
        distance= Distance.COSINE
    )
)

True

In [17]:
vectorPoints = []
for i, vector in enumerate(docEmbeddings):
    obj = PointStruct(
        id = i,
        vector= vector,
        payload={"page":i}
    )
    vectorPoints.append(obj)

client.upsert(
    collection_name="pdf_pages",
    points= vectorPoints
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [24]:
query = "What is the value of RBC and Decreased Levels?"
qVector = embeddings.embed_query(query)

In [25]:
records = client.query_points(collection_name="pdf_pages", query= qVector[0], limit=2).points

In [26]:
top_pages=[]
for record in records:
    top_pages.append(record.payload.get("page"))

top_pages

[0, 6]

In [28]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1")

In [34]:
prompt = [
    {"role":"user",
     "content": [
        {"type":"text", "text":f"Answer only from the context, if not available then say I don't know,my question is:{query}"},
        {"type":"image_url", "image_url": {"url": page_urls[top_pages[0]]}},
        {"type":"image_url", "image_url": {"url": page_urls[top_pages[1]]}}
     ]
    }
]

res= llm.invoke(prompt)

In [35]:
print(res.content)

**RBC value:** The RBC (Red Blood Cell) count in the report is **4.47 mill/mm³**.

**Decreased Levels:** According to the provided context, decreased levels can be caused by:
- Inadequate exposure to sunlight
- Dietary deficiency
- Vitamin D malabsorption
- Severe Hepatocellular disease
- Drugs like Anticonvulsants
- Nephrotic syndrome
